# Retail PySpark pipeline

Star schema in Parquet: transactions partitioned by `sale_month`, plus customer and product dimensions. The executable source is `pyspark_pipeline.py`. This notebook records the run.


In [1]:
from retail_pipeline.session import build_spark
spark = build_spark()
print(spark.version)


Spark 4.3.0  Java 21.0.13.1
UI http://127.0.0.1:4040


## Dataset

Transactions: **1.103 GiB (1.185 GB, 1,184,630,303 bytes)**, 14,244,500 rows, 36 month partitions.

High-entropy `receipt_token` and `payment_ref` columns make the Parquet files large. The revenue pipeline never reads them.


In [2]:
lazy = (
    transactions.filter(sale_month == month)
        .filter(status == 'COMPLETED')
        .withColumn('gross_revenue', round(quantity * unit_price, 2))
)
lazy.count()  # this is the action


transformations 0.0426s, jobs 20 -> 20
explain jobs 20
count() 0.3298s, rows 318813, jobs 22


## Optimized physical plan

Filters on `sale_month` are partition filters. Status, discount, and quantity are pushed into the Parquet scan. Products and customers are broadcast.


In [3]:
monthly.explain(mode='extended')


+- Project [customer_id#156L, product_id#157L, transaction_id#155L, sale_month#171, quantity#160, unit_price#161, discount_pct#162, channel#163, status#165, store_region#166, gross_revenue#388, net_revenue#389, sale_year#390, price_band#391, category#173, subcategory#174, brand#175, segment#183, loyalty_tier#184, customer_region#182]
      :- Project [product_id#157L, transaction_id#155L, customer_id#156L, sale_month#171, quantity#160, unit_price#161, discount_pct#162, channel#163, status#165, store_region#166, gross_revenue#388, net_revenue#389, sale_year#390, price_band#391, category#173, subcategory#174, brand#175]
      :     :- Project [transaction_id#155L, customer_id#156L, product_id#157L, sale_month#171, quantity#160, unit_price#161, discount_pct#162, channel#163, status#165, store_region#166, gross_revenue#388, net_revenue#389, sale_year#390, price_band#391]
      :     :  +- Project [transaction_id#155L, customer_id#156L, product_id#157L, sale_date#159, sale_month#171, quanti

## Cache

The customer rollup shuffles. The second action repeats that shuffle unless the rollup is cached.


In [4]:
customer_value.persist()
customer_value.agg(avg('lifetime_net')).collect()


uncached avg 6.326s
materialize 7.538s
cached avg 0.242s
in-memory plan after cache: True


## Performance analysis

Catalyst pushed the revenue predicates into the Parquet scan and turned the month predicate into a partition filter. The physical plan for `optimized-monthly-write` shows `PushedFilters` with `EqualTo(status, COMPLETED)`, `LessThanOrEqual(discount_pct, 0.5)`, and `GreaterThan(quantity, 0)`, plus `PartitionFilters: [sale_month >= 2023-01]`. The scan's `ReadSchema` keeps the columns the aggregation actually uses and drops `receipt_token`, `payment_ref`, `device_type`, `promo_code`, `payment_method`, and `store_id`. Those token columns are most of the file size, so this is the column-pruning win. A partial `HashAggregate` runs before the shuffle and a final aggregate runs after it. `countDistinct(customer_id)` adds a second exchange. The final plan is an `AdaptiveSparkPlan`: AQE coalesced the shuffle, and the UI shows two `BroadcastHashJoin` nodes (`BuildRight`) rather than a shuffle of the fact table. Whole-stage codegen wraps the scan and the aggregate.

The month filter and the date filter return the same 9,495,016 rows, which is the check that they describe the same window. Filtering `sale_month >= 2023-01` finished in 0.37s with a partition filter and an empty `DataFilters` list, so Spark did not open the 2022 directories. Filtering `sale_date >= 2023-01-01` has `PartitionFilters: []` and a pushed date predicate, and took 0.53s, because those 2022 files are still opened and then row-filtered. A Python UDF on `status` removes `PushedFilters` entirely, forces a `BatchEvalPython` eval, and reads the token columns; on June 2024 that was 1.35s versus 0.13s for the native filter. The join is not the expensive step once the dimensions are broadcast. The naive plan still pushes the same filters, then sort-merge joins both dimensions. The optimized plan broadcasts them. On this laptop the two jobs were close (13.6s for the shuffle joins, 15.4s for the broadcast plan including the Parquet write). Shuffle metrics show 0 remote bytes and 0 spill: local mode moves shuffle blocks on this machine, not across a network. The final plan still records a 1,289 MiB aggregation exchange of the 7.13 million surviving fact rows. That exchange is the real bottleneck, and it is the one a cluster would send over the network. Broadcasting the dimensions avoids doing it two more times.

The pipeline writes the month filter, the status and discount filters, and the narrow column list before either join, and it broadcasts the dimension projections explicitly instead of relying on the 10 MB auto-broadcast threshold. The customer side of the monthly query therefore scans only `customer_id` and `is_active`. The fact table is written in `sale_month` directories so the partition filter matches real paths. Caching is applied to the customer rollup, which is the result that SQL reads again, not to the raw fact table. The uncached average reran the join and shuffle in 6.33s. After `persist()`, the same average took 0.24s, and the segment query's plan starts at `InMemoryTableScan` instead of `FileScan`.



## Model

Logistic regression on a 2.5% sample predicts cancellation from channel, category, segment, loyalty, quantity, price, and discount.


In [5]:
BinaryClassificationEvaluator(metricName='areaUnderROC').evaluate(predictions)


AUC 0.70456360402781
PR AUC 0.2377103949256445
